In [ ]:
#Initialization
import ee, geemap, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive')

try: import jenkspy
except ImportError:
    import subprocess; subprocess.check_call(['pip','install','jenkspy','-q'])
    import jenkspy

try: ee.Initialize(project='geeexercise')
except: ee.Authenticate(); ee.Initialize(project='geeexercise')
print('GEE OK')

In [ ]:
# Configurations
aoi  = ee.FeatureCollection('projects/geeexercise/assets/gebe-island-aoi-land').geometry()
yrs  = list(range(2019, 2026))
crs, scale = 'EPSG:32752', 10

all_12     = ['NDVI','EVI','MSAVI2','BSI','NDWI','NBR','Clay','Ferrous','IronOxide','VV','VH','VV_VH_ratio']
feat_bands = ['NDVI','EVI','MSAVI2','BSI','NDWI','NBR','Clay','Ferrous','VV','VH','VV_VH_ratio']

codes  = [1,2,3,4,5]
lbls   = ['Vegetation','BareSoil','Bush','BuiltUp','Waterbodies']
pal    = ['#1a9641','#d2b48c','#a6d96a','#d7191c','#2b83ba']
c2n    = dict(zip(codes, lbls))

stack_tmpl = 'projects/geeexercise/assets/new-thesis-img/FeatureStack_{yr}'
lulc_tmpl  = 'projects/geeexercise/assets/new-thesis-img/LULC_Gebe_{yr}_RF200trees'

In [ ]:
# Severity component weights
weight = {'ndvi': 0.35, 'bsi': 0.25, 'trans': 0.25, 'persist': 0.15}
assert abs(sum(weight.values()) - 1.0) < 1e-9

# BareSoil + BuiltUp = degraded for persistence scoring
degraded = [2, 4]

# Transition severity lookup: from*10+to -> score [0,1]
trans_lut = {
    12:1.0, 14:1.0, 13:0.5,           # veg losses
    32:0.8, 34:0.8, 31:0.2,           # bush degradation
    24:0.6, 42:0.4,                    # within-mining
    15:0.0, 25:0.0, 35:0.0, 45:0.0,   # -> water
    51:0.2, 52:0.3, 53:0.2, 54:0.4,   # water ->
    11:0.0, 22:0.7, 33:0.1, 44:0.5, 55:0.0,  # stable
}

tier_names = ['Low','Moderate','High','Severe']
tier_pal   = ['#1a9641','#fee08b','#f46d43','#d73027']

rast_dir = 'gee-exports-new/phase-5-new'
csv_dir  = '/content/drive/MyDrive/gee-exports-new/phase-5/csv'
os.makedirs(csv_dir, exist_ok=True)

print(f'Weights: {weight}  sum={sum(weight.values())}')

In [ ]:
aoi_ha = aoi.area(maxError=1).getInfo() / 1e4
print(f'AOI: {aoi_ha:,.2f} ha')

def pull_stack(yr):
    return ee.Image(stack_tmpl.format(yr=yr)).rename(all_12).select(feat_bands).clip(aoi)

def pull_lulc(yr):
    return ee.Image(lulc_tmpl.format(yr=yr)).rename('lulc').clip(aoi)

# Only need 2019 and 2025 stacks for this phase
s19 = pull_stack(2019)
s25 = pull_stack(2025)
lulcs = {yr: pull_lulc(yr) for yr in yrs}

print(f'Stacks loaded: 2019, 2025')
print(f'LULC loaded: {list(lulcs.keys())}')

In [ ]:
# Component 1: NDVI decline (2019-2025), clip negatives. Recovery = no severity
ndvi_dec = s19.select('NDVI').subtract(s25.select('NDVI')).max(0).rename('ndvi_decline')

# Component 2: BSI increase (2019-2025), same logic
bsi_inc  = s25.select('BSI').subtract(s19.select('BSI')).max(0).rename('bsi_increase')

print('NDVI decline and BSI increase computed. Negative value clipped to 0.')

In [ ]:
# Component 3: LULC transition severity via lookup table
# Encode transition as from*10+to, then assign score per code
t_encoded = lulcs[2019].multiply(10).add(lulcs[2025]).toInt16()

trans_sev = ee.Image.constant(0.3).toFloat()
for code, score in trans_lut.items():
    trans_sev = trans_sev.where(t_encoded.eq(code), float(score))
trans_sev = trans_sev.rename('transition_severity').clip(aoi)

print(f'Transition severity image computed. {len(trans_lut)} lookup entries, default=0.3')

In [ ]:
# Component 4: proportion of years pixel was in a degraded class (baresoil or builtup)
# Mean of 7 binary images = fraction of years degraded
def degraded_mask(yr):
    m = ee.Image.constant(0).byte()
    for c in degraded: m = m.Or(lulcs[yr].eq(c))
    return m.toFloat()

persist = (ee.ImageCollection([degraded_mask(yr) for yr in yrs])
             .mean().rename('persistence').clip(aoi))

print(f'Persistence image computed. Degraded classes: {[c2n[c] for c in degraded]}')

In [ ]:
# Min-max normalize to [0,1]
def normalize(img, band, sc=100):
    stats = img.reduceRegion(ee.Reducer.minMax(), aoi, sc, maxPixels=1e9, bestEffort=True).getInfo()
    lo, hi = stats.get(f'{band}_min', 0.0) or 0.0, stats.get(f'{band}_max', 1.0) or 1.0
    rng = hi - lo
    if rng < 1e-10:
        print(f'  WARNING {band}: zero variance, returning zeros')
        return img.multiply(0).rename(band)
    print(f'  {band}: min={lo:.4f}  max={hi:.4f}')
    return img.subtract(lo).divide(rng).clamp(0,1).rename(band).toFloat()

print('Normalizing components...')
ndvi_n  = normalize(ndvi_dec,  'ndvi_decline')
bsi_n   = normalize(bsi_inc,   'bsi_increase')
trans_n = normalize(trans_sev, 'transition_severity')
pers_n  = normalize(persist,   'persistence')

# Weighted sum
severity = (ndvi_n.multiply(weight['ndvi'])
              .add(bsi_n.multiply(weight['bsi']))
              .add(trans_n.multiply(weight['trans']))
              .add(pers_n.multiply(weight['persist']))
              .clamp(0,1).rename('severity_score').toFloat().clip(aoi))

print('Severity score synthesized.')

In [ ]:
# Sample severity score client-side for Jenks computation
print('Sampling 5000 pixels for Jenks...')
samp = (severity.sample(region=aoi, scale=30, numPixels=5000, seed=42, geometries=False)
                .aggregate_array('severity_score').getInfo())
samp = np.array([v for v in samp if v is not None])

breaks = jenkspy.jenks_breaks(samp.tolist(), n_classes=4)
T1, T2, T3 = breaks[1], breaks[2], breaks[3]

print(f'Samples: {len(samp)}  range=[{samp.min():.4f}, {samp.max():.4f}]')
print(f'Jenks breaks: {[round(b,4) for b in breaks]}')
print(f'Tiers:  Low<{T1:.4f}  Moderate<{T2:.4f}  High<{T3:.4f}  Severe<=1.0')

In [ ]:
# 4-tier classification
classified = (severity.lte(T1).multiply(1)
               .add(severity.gt(T1).And(severity.lte(T2)).multiply(2))
               .add(severity.gt(T2).And(severity.lte(T3)).multiply(3))
               .add(severity.gt(T3).multiply(4))
               .rename('severity_class').toUint8()
               .set('T1',T1).set('T2',T2).set('T3',T3))

# Area stats per tier
px = ee.Image.pixelArea().divide(1e4)
tier_ha, tier_pct = {}, {}
print('Computing tier areas...')
for i, nm in enumerate(tier_names, 1):
    ha = px.updateMask(classified.eq(i)).reduceRegion(
             ee.Reducer.sum(), aoi, scale, maxPixels=1e10, bestEffort=True
         ).get('area').getInfo()
    tier_ha[nm]  = round(ha or 0, 2)
    tier_pct[nm] = round(tier_ha[nm] / aoi_ha * 100, 2)
    print(f'  {nm:<10}: {tier_ha[nm]:>8,.2f} ha  ({tier_pct[nm]:.2f}%)')

print(f'  Total classified: {sum(tier_ha.values()):,.2f} ha')

In [ ]:
# Cross-tabulation: severity tier x 2025 LULC class
print('Computing severity x LULC cross-tab (20 calls)...')
xtab = {}
for i, tnm in enumerate(tier_names, 1):
    xtab[tnm] = {}
    for c, lnm in zip(codes, lbls):
        mask = classified.eq(i).And(lulcs[2025].eq(c))
        ha   = px.updateMask(mask).reduceRegion(
                   ee.Reducer.sum(), aoi, scale, maxPixels=1e10, bestEffort=True
               ).get('area').getInfo()
        xtab[tnm][lnm] = round(ha or 0, 2)
    print(f'  {tnm}: ' + '  '.join(f'{n[:4]}={xtab[tnm][n]:.0f}' for n in lbls))

xtab_df = pd.DataFrame(xtab).T
xtab_df.index.name = 'Severity'

In [ ]:
# AOI mean of each normalized component
comp_imgs   = {'NDVI Decline':ndvi_n, 'BSI Increase':bsi_n, 'Transition':trans_n, 'Persistence':pers_n}
comp_bands  = ['ndvi_decline','bsi_increase','transition_severity','persistence']
comp_w      = list(weight.values())
comp_means  = {}

print('Component mean contributions:')
for (nm, img), band, w in zip(comp_imgs.items(), comp_bands, comp_w):
    v = img.reduceRegion(ee.Reducer.mean(), aoi, 100, maxPixels=1e9, bestEffort=True).get(band).getInfo()
    m = float(v) if v else 0.0
    comp_means[nm] = {'weight': w, 'mean': round(m,4), 'contribution': round(m*w,4)}
    print(f'  {nm:<16}: mean={m:.4f}  weight={w:.2f}  contrib={m*w:.4f}')

contrib_df = pd.DataFrame(comp_means).T

In [ ]:
# Sample NDVI-2019 vs severity score for scatter plot
print('Sampling 3000 pixels for scatter plot...')
sc_stack = s19.select('NDVI').addBands(severity).addBands(lulcs[2025].rename('lulc_2025')).toFloat()
sc_list  = (sc_stack.sample(region=aoi, scale=30, numPixels=3000, seed=42, geometries=False)
                    .reduceColumns(ee.Reducer.toList(3), ['NDVI','severity_score','lulc_2025'])
                    .get('list').getInfo())

sc_df = pd.DataFrame(sc_list, columns=['NDVI_2019','severity_score','lulc_2025']).dropna()
sc_df['lulc_2025'] = sc_df['lulc_2025'].astype(int)
sc_df['class']     = sc_df['lulc_2025'].map(c2n)
print(f'Scatter samples: {len(sc_df)}')

In [ ]:
# Chart 1 (severity score histogram)
fig, ax = plt.subplots(figsize=(10,5))
ax.hist(samp, bins=60, color='#5d6d7e', edgecolor='white', lw=0.3, alpha=0.85)
for t, nm, clr in zip([T1,T2,T3], ['Low-Mod','Mod-High','High-Severe'], ['#2ecc71','#f39c12','#e74c3c']):
    ax.axvline(t, color=clr, ls='--', lw=1.8, label=f'{nm}: {t:.3f}')
for x0, x1, c in zip([0,T1,T2,T3],[T1,T2,T3,1.0], tier_pal):
    ax.axvspan(x0, x1, alpha=0.07, color=c)
ax.set(xlabel='Severity Score', ylabel='Pixel Count (sample)',
       title='Severity Score Distribution\n(Jenks Natural Breaks)')
ax.legend(fontsize=8); ax.grid(axis='y', ls='--', alpha=0.35)
plt.tight_layout()
plt.savefig(f'{csv_dir}/chart1_histogram.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Chart 2 (tier area bar)
fig, ax1 = plt.subplots(figsize=(8,5))
ax2 = ax1.twinx()
x = np.arange(len(tier_names))
ha_v  = [tier_ha[t]  for t in tier_names]
pct_v = [tier_pct[t] for t in tier_names]
bars  = ax1.bar(x, ha_v, 0.55, color=tier_pal, edgecolor='white', lw=0.4)
for bar, pct in zip(bars, pct_v):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(ha_v)*0.01,
             f'{bar.get_height():,.0f} ha\n({pct:.1f}%)', ha='center', fontsize=9, fontweight='bold')
ax2.plot(x, pct_v, 'o-', color='#2c3e50', lw=2, ms=8)
ax1.set(xlabel='Severity Tier', ylabel='Area (ha)', title='Severity Area per Tier')
ax1.set_xticks(x); ax1.set_xticklabels(tier_names)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax2.set_ylabel('% of AOI'); ax2.set_ylim(0, max(pct_v)*1.5)
ax1.grid(axis='y', ls='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{csv_dir}/chart2_tier_area.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Chart 3 (Severity tier by LULC)
fig, ax = plt.subplots(figsize=(10,5))
x, bot = np.arange(len(tier_names)), np.zeros(len(tier_names))
for lnm, lclr in zip(lbls, pal):
    vals = np.array([xtab[t][lnm] for t in tier_names])
    ax.bar(x, vals, 0.6, bottom=bot, label=lnm, color=lclr, edgecolor='white', lw=0.4)
    bot += vals
ax.set(xlabel='Severity Tier', ylabel='Area (ha)', title='Severity Tier Composition by 2025 LULC')
ax.set_xticks(x); ax.set_xticklabels(tier_names)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax.legend(fontsize=9); ax.grid(axis='y', ls='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{csv_dir}/chart3_tier_by_lulc.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Chart 4 (component contribution (weight vs weighted mean))
nms   = list(comp_means.keys())
y     = np.arange(len(nms)); h = 0.35
wvals = [comp_means[n]['weight']       for n in nms]
cvals = [comp_means[n]['contribution'] for n in nms]
clrs  = ['#2980b9','#e74c3c','#8e44ad','#27ae60']
fig, ax = plt.subplots(figsize=(11,5))
b1 = ax.barh(y+h/2, wvals, h, label='Assigned weight', color='#bdc3c7', edgecolor='white')
b2 = ax.barh(y-h/2, cvals, h, label='Weighted contribution', color=clrs, edgecolor='white')
ax.set(xlabel='Score (0-1)', title='Severity Index\nComponent Weights vs Weighted Contribution')
ax.set_yticks(y); ax.set_yticklabels(nms)
for bar, v in zip(list(b1)+list(b2), wvals+cvals):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2, f'{v:.3f}', va='center', fontsize=8)
ax.legend(fontsize=9); ax.grid(axis='x', ls='--', alpha=0.35)
ax.set_xlim(0, max(wvals+cvals)*1.3)
plt.tight_layout()
plt.savefig(f'{csv_dir}/chart4_contribution.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Chart 5 (NDVI 2019 vs severity scatter)
fig, ax = plt.subplots(figsize=(10,6))
for lnm, lclr in zip(lbls, pal):
    sub = sc_df[sc_df['class']==lnm]
    ax.scatter(sub['NDVI_2019'], sub['severity_score'], c=lclr, label=lnm,
               alpha=0.45, s=15, edgecolors='none')
for t, nm, clr in zip([T1,T2,T3],['Low-Mod','Mod-High','High-Sev'],['#27ae60','#f39c12','#e74c3c']):
    ax.axhline(t, color=clr, ls='--', lw=1.2, alpha=0.7, label=f'Break {nm}')
ax.set(xlabel='NDVI (2019)', ylabel='Severity Score (2019-2025)',
       title='2019 NDVI vs Final Severity\nHigh-NDVI pixels that degraded should cluster top-right')
ax.legend(fontsize=8, ncol=2); ax.grid(ls='--', alpha=0.3)
ax.set_xlim(-0.1,1.0); ax.set_ylim(-0.02,1.05)
plt.tight_layout()
plt.savefig(f'{csv_dir}/chart5_ndvi_scatter.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# QA
sev_vis = {'min':1,'max':4,'palette':tier_pal}
sev_leg = {f'{nm} ({[0,T1,T2,T3][i]:.3f}-{[T1,T2,T3,1.0][i]:.3f})':c
           for i,(nm,c) in enumerate(zip(tier_names,tier_pal))}

lat, lon = aoi.centroid().coordinates().getInfo()[::-1]

Map = geemap.Map(center=[lat, lon], zoom=11)
Map.add_basemap('SATELLITE')
Map.addLayer(ee.FeatureCollection([ee.Feature(aoi)]).style(
    **{'color':'#FFFFFF','fillColor':'00000000','width':2}), {}, 'AOI')
Map.addLayer(severity,   {'min':0,'max':1,'palette':tier_pal}, 'Severity Score', shown=False)
Map.addLayer(classified, sev_vis, 'Severity (4 tiers)', shown=True)
Map.add_legend(title='Degradation Severity', legend_dict=sev_leg, position='bottomright')
Map.addLayerControl()
Map

In [ ]:
# Export
tasks = []
def export_img(img, name, uint8=False):
    t = ee.batch.Export.image.toDrive(
        image=(img.toUint8() if uint8 else img.toFloat()),
        description=name, folder=rast_dir, fileNamePrefix=name,
        region=aoi, scale=scale, crs=crs, maxPixels=1e10, fileFormat='GeoTIFF')
    t.start(); tasks.append({'name':name,'task':t})

export_img(severity,   'Severity_Score_Continuous_Gebe_2019_2025')
export_img(classified, 'Severity_Classified_4tier_Gebe_2019_2025', uint8=True)
export_img(ndvi_n,     'Component_NDVI_Decline_Norm')
export_img(bsi_n,      'Component_BSI_Increase_Norm')
export_img(trans_n,    'Component_Transition_Severity_Norm')
export_img(pers_n,     'Component_Persistence_Norm')

print(f'{len(tasks)} tasks submitted to Drive/{rast_dir}')

In [ ]:
# Tier area table
tier_df = pd.DataFrame({
    'Tier':      tier_names,
    'Code':      [1,2,3,4],
    'Area_ha':   [tier_ha[t]  for t in tier_names],
    'Pct_AOI':   [tier_pct[t] for t in tier_names],
    'Jenks_lo':  [0.0, T1, T2, T3],
    'Jenks_hi':  [T1, T2, T3, 1.0],
}).set_index('Tier')
tier_df.to_csv(f'{csv_dir}/severity_tier_area.csv')

xtab_df.to_csv(f'{csv_dir}/severity_lulc_crosstab.csv')
contrib_df.to_csv(f'{csv_dir}/component_contributions.csv')

pd.DataFrame({'Threshold':['T1','T2','T3'], 'Value':[round(T1,6),round(T2,6),round(T3,6)]}
             ).to_csv(f'{csv_dir}/jenks_thresholds.csv', index=False)

print(f'CSVs -> {csv_dir}')
print()
print(tier_df.to_string())

In [ ]:
import time as _t

print('Summary')
print('=' * 55)
print(f'Weights: {weight}')
print(f'Jenks T1={T1:.4f}  T2={T2:.4f}  T3={T3:.4f}')
print()
for nm in tier_names:
    print(f'  {nm:<10}: {tier_ha[nm]:>8,.2f} ha  ({tier_pct[nm]:.2f}%)')
print()

In [ ]:
# Monitor export status
STATUS = {'COMPLETED':'[DONE]   ','RUNNING':'[RUNNING]','READY':'[QUEUED] ','FAILED':'[FAILED] '}
print(f'Export status ({_t.strftime("%Y-%m-%d %H:%M")}):')
done = 0
for i, rec in enumerate(tasks, 1):
    st = rec['task'].status(); state = st.get('state','?')
    if state == 'COMPLETED': done += 1
    print(f"  {i:>2}  {rec['name']:<50} {STATUS.get(state,'[?]')} {state}")
    if state == 'FAILED': print(f"      -> {st.get('error_message','no details')}")
print(f'\n{done}/{len(tasks)} completed. Re-run cell to refresh.')